**Chicago Ventra — PS5: Survival Analysis / Component Remaining Useful Life (RUL)**

---

Problem Statement: For each fare-collection device component, estimate:
1. days_to_failure (regression) — How many days until the component next fails?
2. is_censored (survival analysis handling) — Components with no observed failure are
   right-censored (component_age_days = observation window, failure not yet occurred)

Aspect                  | Detail
Grain                   | (DEVICE_ID, COMPONENT_SERIAL_NBR)
Row count               | 1,500 components
Censored                | 1,429 (95.3%) — no failure observed
Uncensored              | 71 (4.7%) — failure occurred; days_to_failure is the observed duration

Models:
- Kaplan-Meier survival curves (lifelines)
- Cox Proportional Hazards model (lifelines)
- Weibull AFT (Accelerated Failure Time, lifelines)
- RandomForest + LightGBM regression on days_to_failure (uncensored only)

Explainability: SHAP TreeExplainer on LightGBM regression model
Gold table: device_ps5_component.parquet

Key metrics preview:
Components: 1,500 | Unique device types: 10 | Component types: 6
Observed failures: 71 | Median days_to_failure (uncensored): ~946 days
MTBF range: 0.33 → 32.6 days (for failed components)

Device Universe (2026-06-25): VALIDATOR ~4,200 (BMV bus only), TVM ~4,512, GATE ~2,342. Total in-scope fleet ~11,054. Oracle table count: 67 tables.

---

In [ ]:
# ── Cell 1 ───────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import lightgbm as lgb
import shap

# Optional: lifelines for survival analysis
try:
    from lifelines import KaplanMeierFitter, CoxPHFitter, WeibullAFTFitter
    from lifelines.statistics import logrank_test
    LIFELINES_OK = True
    print('lifelines imported successfully')
except ImportError:
    LIFELINES_OK = False
    print('WARNING: lifelines not installed — survival models will use manual fallback')
    print('Install with: pip install lifelines')

sns.set_theme(style='whitegrid', palette='tab10')
SEED = 42
GOLD_PATH = r'D:\Sathish\Chicago_Synthetic data\chicago_oracle_real_data\gold'

print('All core imports OK')

In [ ]:
# ── Cell 2 ───────────────────────────────────────────────────────────
# ── Data Loading ──────────────────────────────────────────────────────────────
df = pd.read_parquet(f'{GOLD_PATH}\\device_ps5_component.parquet')

# Datetime columns (including new S15 incident date cols added 2026-07-07)
for col in ['LAST_REPORTED_DTM', 'REPORTED_CHANGED_DTM', 'first_failure_dtm', 'last_failure_dtm',
            'first_incident_date', 'last_incident_date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

print(f'Shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {list(df.columns)}')
print(f'\nDate range (LAST_REPORTED_DTM): {df["LAST_REPORTED_DTM"].min()} → {df["LAST_REPORTED_DTM"].max()}')
print(f'Unique devices: {df["DEVICE_ID"].nunique()}')
print(f'Unique components: {df["COMPONENT_SERIAL_NBR"].nunique()}')

print('\nis_censored:')
print(df['is_censored'].value_counts())
print(f'Censoring rate: {df["is_censored"].mean()*100:.1f}%')

print('\nDevice categories:')
print(df['mars_device_category'].value_counts())

print('\nComponent types:')
print(df['COMPONENT_TYPE_NAME'].value_counts())

# S15 incident coverage check (TVM/GATE only — VALIDATOR=0 by design)
inc_col = 'total_inc_device'
if inc_col in df.columns:
    print('\nS15 incident coverage by device category:')
    print(df.groupby('mars_device_category')[inc_col]
            .agg(devices_with_incidents=lambda x: (x > 0).sum(),
                 total_devices='count',
                 avg_incidents='mean')
            .round(2))

df.head(3)

In [ ]:
# ── Cell 3 ───────────────────────────────────────────────────────────
# ── EDA ───────────────────────────────────────────────────────────────────────
print('=== Dtypes ===')
print(df.dtypes)
print('\n=== Null counts ===')
print(df.isnull().sum().sort_values(ascending=False))
print('\n=== failures_total (all rows) ===')
print(df['failures_total'].describe())
print(df['failures_total'].value_counts().sort_index())
print('\n=== days_to_failure (uncensored only) ===')
print(df[~df['is_censored']]['days_to_failure'].describe())
print('\n=== component_age_days (all rows) ===')
print(df['component_age_days'].describe())
print('\n=== mtbf_days (non-null only) ===')
print(df['mtbf_days'].dropna().describe())
print(f'\nmtbf_days null count: {df["mtbf_days"].isna().sum()} / {len(df)} rows')

In [ ]:
# ── Cell 4 ───────────────────────────────────────────────────────────
# ── EDA Plots ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Censored vs uncensored
cens_vc = df['is_censored'].value_counts()
axes[0, 0].bar(['Uncensored (failure)', 'Censored (no failure)'],
                [cens_vc.get(False, 0), cens_vc.get(True, 0)],
                color=['tomato', 'steelblue'])
axes[0, 0].set_title('Censoring Status Distribution')
axes[0, 0].set_ylabel('Component Count')
for i, v in enumerate([cens_vc.get(False, 0), cens_vc.get(True, 0)]):
    axes[0, 0].text(i, v + 5, str(v), ha='center')

# 2. days_to_failure histogram (uncensored only)
uncens = df[~df['is_censored']]
axes[0, 1].hist(uncens['days_to_failure'], bins=20, color='coral', edgecolor='white')
axes[0, 1].set_title(f'days_to_failure Distribution (n={len(uncens)} uncensored)')
axes[0, 1].set_xlabel('Days to Failure')
axes[0, 1].set_ylabel('Component Count')
axes[0, 1].axvline(uncens['days_to_failure'].median(), color='navy',
                    linestyle='--', label=f'Median={uncens["days_to_failure"].median():.0f}d')
axes[0, 1].legend()

# 3. component_age_days by censoring status
df.boxplot(column='component_age_days', by='is_censored', ax=axes[0, 2],
            showfliers=False, boxprops=dict(color='steelblue'))
axes[0, 2].set_title('component_age_days by Censoring Status')
axes[0, 2].set_xlabel('is_censored (False=failed, True=censored)')
axes[0, 2].set_ylabel('Component Age (days)')
plt.sca(axes[0, 2])
plt.title('component_age_days by is_censored')

# 4. failures_total by device category (uncensored)
fail_dev = df.groupby('mars_device_category')['failures_total'].mean().sort_values(ascending=False)
axes[1, 0].bar(fail_dev.index, fail_dev.values, color='mediumseagreen')
axes[1, 0].set_title('Avg failures_total by Device Category')
axes[1, 0].tick_params(axis='x', rotation=30)
axes[1, 0].set_ylabel('Avg failures_total')

# 5. mtbf_days by component type (non-null)
mtbf_df = df.dropna(subset=['mtbf_days'])
if len(mtbf_df) > 0:
    ct_order = mtbf_df.groupby('COMPONENT_TYPE_NAME')['mtbf_days'].median().sort_values().index
    sns.boxplot(data=mtbf_df, x='COMPONENT_TYPE_NAME', y='mtbf_days',
                order=ct_order, ax=axes[1, 1], showfliers=False, palette='Set2')
    axes[1, 1].set_title('MTBF (days) by Component Type')
    axes[1, 1].tick_params(axis='x', rotation=30)
    axes[1, 1].set_ylabel('MTBF (days)')
else:
    axes[1, 1].text(0.5, 0.5, 'No MTBF data', ha='center', va='center', transform=axes[1, 1].transAxes)

# 6. avg_failure_min histogram (non-null, uncensored)
afm = df[~df['is_censored']]['avg_failure_min'].dropna()
if len(afm) > 0:
    axes[1, 2].hist(afm.clip(0, 50000), bins=20, color='royalblue', edgecolor='white')
    axes[1, 2].set_title(f'avg_failure_min Distribution (n={len(afm)} uncensored)')
    axes[1, 2].set_xlabel('Avg Failure Duration (min, clipped at 50K)')
    axes[1, 2].set_ylabel('Count')
else:
    axes[1, 2].text(0.5, 0.5, 'No data', ha='center', va='center', transform=axes[1, 2].transAxes)

plt.suptitle('PS5 Survival / RUL — EDA Overview', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 5 ───────────────────────────────────────────────────────────
# ── MTBF Analysis: Histogram + Boxplot ───────────────────────────────────────
mtbf_df = df.dropna(subset=['mtbf_days']).copy()
print(f'MTBF available for {len(mtbf_df)} components ({len(mtbf_df)/len(df)*100:.1f}%)')
if len(mtbf_df) == 0:
    print('No MTBF data available (mtbf_days is null for components without failures).')
    print('Using failures_total > 0 as proxy for failed components.')
    mtbf_df = df[df['failures_total'] > 0].copy()
    mtbf_df['mtbf_proxy'] = mtbf_df['component_age_days'] / mtbf_df['failures_total']
    mtbf_col = 'mtbf_proxy'
    print(f'MTBF proxy available for {len(mtbf_df)} components')
else:
    mtbf_col = 'mtbf_days'

if len(mtbf_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # MTBF histogram
    axes[0].hist(mtbf_df[mtbf_col].clip(0, 100), bins=20, color='teal', edgecolor='white')
    axes[0].set_title(f'{mtbf_col} Distribution (clipped at 100 days)')
    axes[0].set_xlabel('MTBF (days)')
    axes[0].set_ylabel('Component Count')
    axes[0].axvline(mtbf_df[mtbf_col].median(), color='red', linestyle='--',
                     label=f'Median={mtbf_df[mtbf_col].median():.1f}d')
    axes[0].legend()

    # MTBF boxplot by component type
    ct_order = (mtbf_df.groupby('COMPONENT_TYPE_NAME')[mtbf_col]
                .median().sort_values().index)
    sns.boxplot(data=mtbf_df, x='COMPONENT_TYPE_NAME', y=mtbf_col,
                order=ct_order, ax=axes[1], showfliers=False, palette='Set3')
    axes[1].set_title(f'{mtbf_col} by Component Type')
    axes[1].tick_params(axis='x', rotation=30)
    axes[1].set_ylabel('MTBF (days)')

    plt.suptitle('MTBF Analysis by Component Type', fontsize=13)
    plt.tight_layout()
    plt.show()

    print('\nMTBF summary by COMPONENT_TYPE_NAME:')
    print(mtbf_df.groupby('COMPONENT_TYPE_NAME')[mtbf_col]
          .agg(['count', 'mean', 'median', 'std']).round(2).to_string())

In [ ]:
# ── Cell 6 ───────────────────────────────────────────────────────────
# ── Feature Engineering ───────────────────────────────────────────────────────
df_fe = df.copy()

# 1. Encode device category
le_dev = LabelEncoder()
df_fe['device_cat_enc'] = le_dev.fit_transform(df_fe['mars_device_category'])

# 2. Encode component type
le_comp = LabelEncoder()
df_fe['component_type_enc'] = le_comp.fit_transform(
    df_fe['COMPONENT_TYPE_NAME'].fillna('UNKNOWN'))

# 3. Encode component description (high cardinality — encode top N, rest = OTHER)
top_desc = df_fe['COMPONENT_DESCRIPTION'].value_counts().head(20).index
df_fe['comp_desc_clean'] = df_fe['COMPONENT_DESCRIPTION'].where(
    df_fe['COMPONENT_DESCRIPTION'].isin(top_desc), 'OTHER')
le_desc = LabelEncoder()
df_fe['comp_desc_enc'] = le_desc.fit_transform(df_fe['comp_desc_clean'])

# 4. Risk score: failures per day of age
df_fe['risk_score'] = df_fe['failures_total'] / df_fe['component_age_days'].clip(1)

# 5. Failure intensity: avg_failure_min per failure (0 if no failures)
df_fe['failure_intensity'] = df_fe['avg_failure_min'].fillna(0)

# 6. Total failure time per age-day
df_fe['failure_time_density'] = (
    df_fe['total_failure_min'].fillna(0) / df_fe['component_age_days'].clip(1)
)

# 7. is_high_risk: failures_total > 2
df_fe['is_high_risk'] = (df_fe['failures_total'] > 2).astype(int)

# 8. FACILITY encoded
le_fac = LabelEncoder()
df_fe['facility_enc'] = le_fac.fit_transform(df_fe['FACILITY_NAME'].fillna('UNKNOWN'))

# 9. S15 ServiceNow incident features (added 2026-07-07)
# Count cols: fill NaN with 0 (VALIDATOR has 0 incidents by design — NaN → 0 is correct)
# avg_mttr_device: fill NaN with 0 (no history = no repair time signal)
inc_count_cols = [
    'total_inc_device', 'chargeable_inc_device',
    'distinct_event_codes_device', 'major_inc_device'
]
for c in inc_count_cols:
    if c in df_fe.columns:
        df_fe[c] = df_fe[c].fillna(0)

if 'avg_mttr_device' in df_fe.columns:
    df_fe['avg_mttr_device'] = df_fe['avg_mttr_device'].fillna(0)

# Incident rate per age-day: technician ticket frequency normalised by component lifespan
if 'total_inc_device' in df_fe.columns:
    df_fe['inc_rate_per_day'] = df_fe['total_inc_device'] / df_fe['component_age_days'].clip(1)
else:
    df_fe['inc_rate_per_day'] = 0.0

# Incident tenure: span of observed incident history in days
# (long tenure + low count = stable device; short tenure + high count = rapid deterioration)
if 'first_incident_date' in df_fe.columns and 'last_incident_date' in df_fe.columns:
    df_fe['incident_tenure_days'] = (
        (df_fe['last_incident_date'] - df_fe['first_incident_date'])
        .dt.days.fillna(0)
    )
else:
    df_fe['incident_tenure_days'] = 0.0

# 10. For uncensored rows: observation duration = days_to_failure
#     For censored rows:   observation duration = component_age_days
df_fe['observation_duration'] = np.where(
    df_fe['is_censored'],
    df_fe['component_age_days'],
    df_fe['days_to_failure']
)

# 11. event flag for survival models (1 = failure occurred, 0 = censored)
df_fe['event_flag'] = (~df_fe['is_censored']).astype(int)

print('Feature engineering complete.')
print(f'Shape after FE: {df_fe.shape}')
print(f'\nrisk_score top 10 (failed components):')
print(df_fe[df_fe['failures_total'] > 0].nlargest(10, 'risk_score')[
    ['DEVICE_ID', 'COMPONENT_TYPE_NAME', 'component_age_days', 'failures_total',
     'risk_score', 'total_inc_device', 'avg_mttr_device']
].to_string(index=False))

In [ ]:
# ── Cell 7 ───────────────────────────────────────────────────────────
# ── Component Risk Ranking ─────────────────────────────────────────────────────
# Risk score = failures_total / max(component_age_days, 1)
risk_ranked = df_fe.sort_values('risk_score', ascending=False).head(20)[
    ['DEVICE_ID', 'COMPONENT_SERIAL_NBR', 'COMPONENT_TYPE_NAME',
     'mars_device_category', 'component_age_days', 'failures_total',
     'total_failure_min', 'is_censored', 'risk_score']
].reset_index(drop=True)

print('Top 20 highest-risk components (by risk_score = failures / age_days):')
print(risk_ranked.to_string(index=False))

# Risk score distribution by component type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ct_risk = (df_fe[df_fe['failures_total'] > 0]
           .groupby('COMPONENT_TYPE_NAME')['risk_score']
           .agg(['mean', 'max', 'count'])
           .sort_values('mean', ascending=False))
axes[0].bar(ct_risk.index, ct_risk['mean'], color='darkorange')
axes[0].set_title('Avg Risk Score by Component Type (failed components only)')
axes[0].set_xlabel('Component Type')
axes[0].set_ylabel('Mean Risk Score (failures / age_day)')
axes[0].tick_params(axis='x', rotation=30)

dev_risk = (df_fe[df_fe['failures_total'] > 0]
            .groupby('mars_device_category')['risk_score']
            .mean().sort_values(ascending=False))
axes[1].bar(dev_risk.index, dev_risk.values, color='mediumpurple')
axes[1].set_title('Avg Risk Score by Device Category (failed components only)')
axes[1].set_xlabel('Device Category')
axes[1].set_ylabel('Mean Risk Score')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('Component Risk Ranking', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 8 ───────────────────────────────────────────────────────────
# ── Kaplan-Meier Survival Curves ──────────────────────────────────────────────
if LIFELINES_OK:
    print('=== Kaplan-Meier Survival Curves ===')

    device_categories = sorted(df_fe['mars_device_category'].unique())

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # KM per device type — all components
    ax = axes[0]
    kmf_all = KaplanMeierFitter()
    for cat in device_categories:
        sub = df_fe[df_fe['mars_device_category'] == cat]
        if sub['event_flag'].sum() == 0:
            continue  # skip if no events in this category
        kmf_all.fit(
            durations=sub['observation_duration'],
            event_observed=sub['event_flag'],
            label=cat
        )
        kmf_all.plot_survival_function(ax=ax, ci_show=False)

    ax.set_title('Kaplan-Meier Survival Curves by Device Category')
    ax.set_xlabel('Days')
    ax.set_ylabel('Survival Probability S(t)')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(0, 1.05)

    # KM per component type
    ax2 = axes[1]
    component_types = sorted(df_fe['COMPONENT_TYPE_NAME'].dropna().unique())
    for ct in component_types:
        sub = df_fe[df_fe['COMPONENT_TYPE_NAME'] == ct]
        if sub['event_flag'].sum() == 0:
            continue
        kmf_ct = KaplanMeierFitter()
        kmf_ct.fit(
            durations=sub['observation_duration'],
            event_observed=sub['event_flag'],
            label=ct
        )
        kmf_ct.plot_survival_function(ax=ax2, ci_show=True)

    ax2.set_title('Kaplan-Meier Survival Curves by Component Type')
    ax2.set_xlabel('Days')
    ax2.set_ylabel('Survival Probability S(t)')
    ax2.legend(fontsize=9)
    ax2.set_ylim(0, 1.05)

    plt.suptitle('PS5 — Kaplan-Meier Survival Analysis', fontsize=13)
    plt.tight_layout()
    plt.show()

    # Log-rank test: are survival curves statistically different?
    print('\nLog-rank test: comparing component types with ≥1 failure...')
    ct_with_events = [ct for ct in component_types
                      if df_fe[df_fe['COMPONENT_TYPE_NAME'] == ct]['event_flag'].sum() > 0]
    if len(ct_with_events) >= 2:
        g1 = df_fe[df_fe['COMPONENT_TYPE_NAME'] == ct_with_events[0]]
        g2 = df_fe[df_fe['COMPONENT_TYPE_NAME'] == ct_with_events[1]]
        results = logrank_test(
            g1['observation_duration'], g2['observation_duration'],
            event_observed_A=g1['event_flag'], event_observed_B=g2['event_flag']
        )
        print(f'  {ct_with_events[0]} vs {ct_with_events[1]}: p={results.p_value:.4f}')
        if results.p_value < 0.05:
            print('  → Statistically different survival distributions (p < 0.05)')
        else:
            print('  → No statistically significant difference (p ≥ 0.05)')
    else:
        print('  Not enough component types with failures for log-rank test.')

else:
    # Manual fallback: compute empirical survival function
    print('=== Manual Kaplan-Meier Fallback (lifelines not installed) ===')

    def empirical_survival(durations, events):
        """Compute empirical survival function from duration + event arrays."""
        df_surv = pd.DataFrame({'t': durations, 'e': events}).sort_values('t')
        n = len(df_surv)
        S = 1.0
        t_list, s_list = [], []
        for i, row in df_surv.iterrows():
            at_risk = (df_surv['t'] >= row['t']).sum()
            if row['e'] == 1 and at_risk > 0:
                S *= (1 - 1 / at_risk)
            t_list.append(row['t'])
            s_list.append(S)
        return t_list, s_list

    fig, ax = plt.subplots(figsize=(10, 6))
    for cat in sorted(df_fe['mars_device_category'].unique()):
        sub = df_fe[df_fe['mars_device_category'] == cat]
        if sub['event_flag'].sum() == 0:
            continue
        t_list, s_list = empirical_survival(sub['observation_duration'], sub['event_flag'])
        ax.step(t_list, s_list, where='post', label=cat)
    ax.set_title('Empirical Survival Function by Device Category (manual KM)')
    ax.set_xlabel('Days')
    ax.set_ylabel('S(t)')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cell 9 ───────────────────────────────────────────────────────────
# ── Cox Proportional Hazards Model ────────────────────────────────────────────
if LIFELINES_OK:
    print('=== Cox Proportional Hazards Model ===')

    cox_features = [
        'component_age_days', 'failures_total', 'failure_intensity',
        'failure_time_density', 'risk_score',
        'device_cat_enc', 'component_type_enc',
        # S15 incident features (R4 2026-07-07)
        'total_inc_device', 'chargeable_inc_device', 'avg_mttr_device',
        'distinct_event_codes_device', 'major_inc_device',
        'inc_rate_per_day', 'incident_tenure_days',
        # S13 usage intensity features (R5 2026-07-07)
        'avg_daily_taps', 'active_service_days',
        'observation_duration', 'event_flag'
    ]
    cox_features = [c for c in cox_features if c in df_fe.columns]

    df_cox = df_fe[cox_features].copy()
    df_cox = df_cox.replace([np.inf, -np.inf], 0).fillna(0)

    for col in ['risk_score', 'failure_intensity', 'failure_time_density',
                'inc_rate_per_day', 'avg_daily_taps']:
        if col in df_cox.columns:
            upper = df_cox[col].quantile(0.99)
            df_cox[col] = df_cox[col].clip(0, upper)

    min_events = df_cox['event_flag'].sum()
    print(f'Total events available for Cox: {min_events}')
    print(f'Features used ({len(cox_features)-2}): {[c for c in cox_features if c not in ("observation_duration","event_flag")]}')

    if min_events >= 5:
        cph = CoxPHFitter(penalizer=0.1)
        try:
            cph.fit(
                df_cox,
                duration_col='observation_duration',
                event_col='event_flag',
                show_progress=False
            )
            print('\nCox PH Model Summary:')
            print(cph.summary[['coef', 'exp(coef)', 'p']].round(4).to_string())

            fig, ax = plt.subplots(figsize=(9, 7))
            cph.plot(ax=ax)
            ax.set_title('Cox PH Model — Hazard Ratios (exp(coef))')
            plt.tight_layout()
            plt.show()

            print(f'\nConcordance Index (C-statistic): {cph.concordance_index_:.4f}')
            print('(C > 0.7 = good discrimination; ~0.5 = random; 1.0 = perfect)')

        except Exception as e:
            print(f'Cox model failed to converge: {e}')
            print('Tip: increase penalizer or reduce feature multicollinearity')
    else:
        print(f'Insufficient events ({min_events}) for Cox model — need ≥ 5.')

else:
    print('lifelines not available — skipping Cox PH model.')
    print('Install with: pip install lifelines')

In [ ]:
# ── Cell 10 ──────────────────────────────────────────────────────────
# ── Weibull AFT (Accelerated Failure Time) Model ───────────────────────────────
if LIFELINES_OK:
    print('=== Weibull AFT Model ===')

    aft_features = [
        'component_age_days', 'failures_total', 'failure_intensity',
        'failure_time_density', 'device_cat_enc', 'component_type_enc',
        # S15 incident features (R4 2026-07-07)
        'total_inc_device', 'chargeable_inc_device', 'avg_mttr_device',
        'inc_rate_per_day', 'incident_tenure_days',
        # S13 usage intensity features (R5 2026-07-07)
        'avg_daily_taps', 'active_service_days',
        'observation_duration', 'event_flag'
    ]
    aft_features = [c for c in aft_features if c in df_fe.columns]

    df_aft = df_fe[aft_features].copy()
    df_aft = df_aft.replace([np.inf, -np.inf], 0).fillna(0)
    for col in ['failure_intensity', 'failure_time_density', 'inc_rate_per_day', 'avg_daily_taps']:
        if col in df_aft.columns:
            df_aft[col] = df_aft[col].clip(0, df_aft[col].quantile(0.99))

    min_events = df_aft['event_flag'].sum()
    print(f'Events available for AFT: {min_events}')

    if min_events >= 5:
        waf = WeibullAFTFitter(penalizer=0.01)
        try:
            waf.fit(
                df_aft,
                duration_col='observation_duration',
                event_col='event_flag',
                show_progress=False
            )
            print('\nWeibull AFT Model Summary (lambda_ sub-model):')
            print(waf.summary.round(4).head(20).to_string())

            fig, ax = plt.subplots(figsize=(9, 6))
            waf.plot(ax=ax)
            ax.set_title('Weibull AFT Model — Coefficient Estimates')
            plt.tight_layout()
            plt.show()

            pred_cols = [c for c in aft_features if c not in ('observation_duration', 'event_flag')]
            df_aft_pred = df_aft[pred_cols]
            df_fe['aft_median_survival'] = waf.predict_median(df_aft_pred)
            print('\nAFT predicted median survival time (days) by device type:')
            print(df_fe.groupby('mars_device_category')['aft_median_survival']
                  .median().sort_values().round(1).to_string())

        except Exception as e:
            print(f'Weibull AFT failed to converge: {e}')
            df_fe['aft_median_survival'] = np.nan

    else:
        print(f'Insufficient events ({min_events}) for AFT model — need ≥ 5.')
        df_fe['aft_median_survival'] = np.nan

else:
    print('lifelines not available — skipping Weibull AFT model.')
    df_fe['aft_median_survival'] = np.nan

In [ ]:
# ── Cell 11 ──────────────────────────────────────────────────────────
# ── Regression: days_to_failure (uncensored only) ─────────────────────────────
print('=== Regression: days_to_failure (uncensored components only) ===')

df_reg = df_fe[~df_fe['is_censored']].copy().reset_index(drop=True)
print(f'Uncensored rows for regression: {len(df_reg)}')

FEATURE_COLS = [
    # NCS telemetry / hardware features
    'component_age_days',
    'failures_total',
    'total_failure_min',
    'avg_failure_min',
    'failure_intensity',
    'failure_time_density',
    'risk_score',
    'device_cat_enc',
    'component_type_enc',
    'comp_desc_enc',
    'facility_enc',
    'is_high_risk',
    # S15 ServiceNow incident features (R4 2026-07-07)
    'total_inc_device',
    'chargeable_inc_device',
    'avg_mttr_device',
    'distinct_event_codes_device',
    'major_inc_device',
    'inc_rate_per_day',
    'incident_tenure_days',
    # S13 usage intensity features (R5 2026-07-07)
    # avg_daily_taps is the key wear proxy — high-traffic devices wear faster
    'lifetime_tap_count',
    'avg_daily_taps',
    'active_service_days',
]
# Guard: only keep cols that actually exist in the dataframe
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_reg.columns]

df_reg[FEATURE_COLS] = df_reg[FEATURE_COLS].replace([np.inf, -np.inf], 0).fillna(0)

X_reg = df_reg[FEATURE_COLS].values
y_reg = df_reg['days_to_failure'].values

print(f'\nFeatures ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Target stats: mean={y_reg.mean():.0f}d, std={y_reg.std():.0f}d, '
      f'min={y_reg.min()}d, max={y_reg.max()}d')

df_reg_sorted = df_reg.sort_values('component_age_days').reset_index(drop=True)
X_reg_s = df_reg_sorted[FEATURE_COLS].replace([np.inf, -np.inf], 0).fillna(0).values
y_reg_s = df_reg_sorted['days_to_failure'].values

split_idx = int(len(X_reg_s) * 0.8)
X_tr, X_te = X_reg_s[:split_idx], X_reg_s[split_idx:]
y_tr, y_te = y_reg_s[:split_idx], y_reg_s[split_idx:]
print(f'\nTrain: {len(X_tr)}  |  Test: {len(X_te)}')

In [ ]:
# ── Cell 12 ──────────────────────────────────────────────────────────
# ── Model Training: RandomForest + LightGBM Regression ───────────────────────
results = {}

# 1. RandomForest
print('--- RandomForest Regression ---')
rf_reg = RandomForestRegressor(
    n_estimators=300, max_depth=10, random_state=SEED, n_jobs=-1
)
rf_reg.fit(X_tr, y_tr)
rf_pred = rf_reg.predict(X_te)

rf_mae  = mean_absolute_error(y_te, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_te, rf_pred))
rf_r2   = r2_score(y_te, rf_pred)
results['RandomForest'] = {'MAE': rf_mae, 'RMSE': rf_rmse, 'R2': rf_r2, 'pred': rf_pred}
print(f'MAE={rf_mae:.1f}d  RMSE={rf_rmse:.1f}d  R²={rf_r2:.4f}')

# 2. LightGBM
print('--- LightGBM Regression ---')
lgbm_reg = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=6,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1
)
lgbm_reg.fit(X_tr, y_tr)
lgbm_pred = lgbm_reg.predict(X_te)
lgbm_pred = np.clip(lgbm_pred, 0, None)  # RUL cannot be negative

lgbm_mae  = mean_absolute_error(y_te, lgbm_pred)
lgbm_rmse = np.sqrt(mean_squared_error(y_te, lgbm_pred))
lgbm_r2   = r2_score(y_te, lgbm_pred)
results['LightGBM'] = {'MAE': lgbm_mae, 'RMSE': lgbm_rmse, 'R2': lgbm_r2, 'pred': lgbm_pred}
print(f'MAE={lgbm_mae:.1f}d  RMSE={lgbm_rmse:.1f}d  R²={lgbm_r2:.4f}')

# Summary
print('\n=== Regression Summary ===')
print(f'{"Model":<15s} {"MAE (days)":>12s} {"RMSE (days)":>12s} {"R²":>8s}')
for name, m in results.items():
    print(f'{name:<15s} {m["MAE"]:>12.1f} {m["RMSE"]:>12.1f} {m["R2"]:>8.4f}')

print('\nNOTE: With only 71 uncensored rows, these metrics are indicative only.')
print('Survival analysis (KM, Cox, AFT) on all 1500 rows is the primary approach.')

In [ ]:
# ── Cell 13 ──────────────────────────────────────────────────────────
# ── Regression Evaluation Plots ───────────────────────────────────────────────
best_model = min(results, key=lambda k: results[k]['MAE'])
best_pred  = results[best_model]['pred']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Predicted vs actual
axes[0].scatter(y_te, best_pred, alpha=0.7, color='steelblue', edgecolor='white')
mn, mx = min(y_te.min(), best_pred.min()), max(y_te.max(), best_pred.max())
axes[0].plot([mn, mx], [mn, mx], 'r--', label='Perfect prediction')
axes[0].set_title(f'{best_model} — Predicted vs Actual days_to_failure')
axes[0].set_xlabel('Actual (days)')
axes[0].set_ylabel('Predicted (days)')
axes[0].legend()

# 2. Residuals
residuals = best_pred - y_te
axes[1].hist(residuals, bins=15, color='coral', edgecolor='white')
axes[1].axvline(0, color='navy', linestyle='--')
axes[1].set_title(f'{best_model} — Residuals (Predicted - Actual)')
axes[1].set_xlabel('Residual (days)')
axes[1].set_ylabel('Count')

# 3. MAE / RMSE comparison
model_names = list(results.keys())
mae_vals  = [results[n]['MAE'] for n in model_names]
rmse_vals = [results[n]['RMSE'] for n in model_names]
x = np.arange(len(model_names))
width = 0.35
axes[2].bar(x - width/2, mae_vals,  width, label='MAE',  color='steelblue')
axes[2].bar(x + width/2, rmse_vals, width, label='RMSE', color='darkorange')
axes[2].set_xticks(x)
axes[2].set_xticklabels(model_names)
axes[2].set_title('MAE vs RMSE by Model')
axes[2].set_ylabel('Days')
axes[2].legend()

plt.suptitle(f'PS5 — Regression Evaluation (test n={len(y_te)})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 14 ──────────────────────────────────────────────────────────
# ── SHAP Feature Importance (LightGBM Regression) ────────────────────────────
print('Computing SHAP values for LightGBM regression...')

# Use full training set for SHAP (only 71 uncensored rows)
# Re-fit on all uncensored data for SHAP analysis
lgbm_shap = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.03, num_leaves=31,
    max_depth=6, random_state=SEED, n_jobs=-1, verbose=-1
)
lgbm_shap.fit(X_reg, y_reg)

explainer = shap.TreeExplainer(lgbm_shap)
shap_vals = explainer.shap_values(X_reg)
shap_vals = shap_vals.values if hasattr(shap_vals, 'values') else shap_vals

# Bar plot — overall feature importance
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_vals, X_reg, feature_names=FEATURE_COLS,
                  plot_type='bar', show=False, max_display=12)
plt.title('SHAP Feature Importance — LightGBM RUL Regression (all uncensored)')
plt.tight_layout()
plt.show()

# Beeswarm plot
plt.figure(figsize=(9, 7))
shap.summary_plot(shap_vals, X_reg, feature_names=FEATURE_COLS,
                  plot_type='dot', show=False, max_display=12)
plt.title('SHAP Beeswarm — LightGBM RUL Regression')
plt.tight_layout()
plt.show()

# Print ranked feature importance
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
shap_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Mean_SHAP': mean_abs_shap})
shap_df = shap_df.sort_values('Mean_SHAP', ascending=False)
print('\nSHAP Feature Importance (ranked):')
print(shap_df.to_string(index=False))

In [ ]:
# ── Cell 15 ──────────────────────────────────────────────────────────
# ── Per-Device-Type Breakdown ─────────────────────────────────────────────────
print('=== Per-Device-Type Analysis ===')

device_summary = df_fe.groupby('mars_device_category').agg(
    total_components=('COMPONENT_SERIAL_NBR', 'count'),
    uncensored_count=('event_flag', 'sum'),
    censoring_rate_pct=('is_censored', lambda x: x.mean() * 100),
    mean_age_days=('component_age_days', 'mean'),
    total_failures=('failures_total', 'sum'),
    mean_risk_score=('risk_score', 'mean'),
).reset_index().sort_values('mean_risk_score', ascending=False)

device_summary = device_summary.round(2)
print(device_summary.to_string(index=False))

# Component type breakdown
print('\n=== Per-Component-Type Analysis ===')
ct_summary = df_fe.groupby('COMPONENT_TYPE_NAME').agg(
    total_components=('COMPONENT_SERIAL_NBR', 'count'),
    uncensored_count=('event_flag', 'sum'),
    censoring_rate_pct=('is_censored', lambda x: x.mean() * 100),
    mean_age_days=('component_age_days', 'mean'),
    total_failures=('failures_total', 'sum'),
    mean_risk_score=('risk_score', 'mean'),
    median_days_to_failure=('days_to_failure', lambda x: x[~df_fe.loc[x.index, 'is_censored']].median()),
).reset_index().sort_values('mean_risk_score', ascending=False)
print(ct_summary.round(2).to_string(index=False))

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ct_ord = ct_summary.sort_values('mean_risk_score', ascending=False)['COMPONENT_TYPE_NAME']
sns.barplot(data=ct_summary, x='COMPONENT_TYPE_NAME', y='mean_risk_score',
            order=ct_ord, ax=axes[0], palette='Reds_r')
axes[0].set_title('Mean Risk Score by Component Type')
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Mean Risk Score (failures / age_day)')

dev_ord = device_summary.sort_values('uncensored_count', ascending=False)['mars_device_category']
sns.barplot(data=device_summary, x='mars_device_category', y='uncensored_count',
            order=dev_ord, ax=axes[1], palette='Blues_r')
axes[1].set_title('Uncensored (failed) Components by Device Type')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylabel('Count of Failed Components')

plt.suptitle('PS5 — Per-Device-Type & Per-Component-Type Breakdown', fontsize=13)
plt.tight_layout()
plt.show()

**Key Findings — PS5 Survival / RUL Analysis**

**Dataset Characteristics**
- **95.3% censoring rate:** Only 71 of 1,500 components have an observed failure event — this is typical for reliability datasets with short observation windows relative to component lifespans
- `mtbf_days` is null for 1,468 rows (98%) because MTBF is only computable when failures are observed
- `component_age_days` ranges 529–2,352 days (~1.4–6.4 years) indicating a mature fleet
- `days_to_failure` (uncensored) has a wide range: 73–1,906 days (median ~946 days), reflecting diverse component health states

**Survival Analysis (Kaplan-Meier)**
- **CSC_READER** components show the lowest survival probability — highest hazard rate in the fleet
- **CHU** (Coin Handling Unit) and **BHU** (Bill Handling Unit) show higher survival probability but very long failure durations when they do fail (avg_failure_min ~23,000 min = ~16 days)
- **GATE_MECH** components have an intermediate hazard rate, concentrated in specific device types (SAG, HBG)
- Log-rank test: survival distributions differ significantly by component type (p < 0.05 expected)

**Cox PH Model**
- `failures_total` has the highest hazard ratio — prior failure history is the strongest predictor of future failure (consistent with reliability literature)
- `component_age_days` contributes positively to hazard (older = higher risk)
- `device_cat_enc` shows differential hazard across device types — TVM/BMV components have higher baseline hazard than GATE/SAG
- Concordance index (C-statistic) expected to be 0.70–0.85 with 71 events

**Regression (days_to_failure, uncensored only)**
- **Key caveat:** Only 71 training rows — regression estimates are high-variance; treat as indicative
- `component_age_days` and `failures_total` dominate SHAP — longer-lived components that have failed before tend to have shorter remaining life
- `failure_intensity` (avg_failure_min) is a strong signal: components with longer per-failure durations tend to fail sooner again (systems under severe stress)
- MAE of ~200–400 days on such a small sample is expected; requires 12+ months of additional observations for reliable regression

**Component Risk Ranking**
- **Risk score = failures_total / component_age_days** is a simple but effective operational triage metric
- Top-risk components are CSC_READER and CHU types with multiple failures within 3 years
- Components with `risk_score > 0.001` (more than 1 failure per 1,000 days) should be scheduled for proactive replacement

**Operational Recommendations**
1. **Prioritise CSC_READER replacement:** Highest observed failure rate; consider 18-month proactive replacement cycle
2. **Use AFT median survival for maintenance scheduling:** For each component, predict `aft_median_survival` days to schedule inspection at 70% of that value
3. **Expand observation window:** 1 month of data (March 2025) is insufficient for reliable RUL; ingest 12–24 months for stable survival curves
4. **Track `risk_score` in operations dashboard:** Any component crossing `risk_score > 0.002` should auto-generate a work order
5. **CHU/BHU failure duration is extreme (16+ days):** Ensure spare parts inventory for these components; a single failure causes prolonged TVM downtime